# Lezione 2 — Matplotlib: grafici a linee e ad area

## Perché questo ti riguarda?

Una delle domande più comuni in qualsiasi analisi dei dati è: "come è cambiato questo nel tempo?". Salgono le vendite? Scendono i casi? È esploso qualcosa in un anno preciso? Il **grafico a linee** è la risposta visiva più diretta a questo tipo di domanda, ed è probabilmente il primo grafico che disegnerai in qualsiasi progetto reale.

In questa lezione useremo `pandas` e `matplotlib` per disegnare tendenze con il dataset dell'immigrazione verso il Canada. Vedremo un solo paese, più paesi contemporaneamente, e una variante del grafico a linee — il **grafico ad area** — che oltre a mostrare la tendenza, riempie lo spazio sotto la linea per dare la sensazione di volume accumulato.

## Due modi di disegnare: l'appunto veloce e il design a strati

Matplotlib offre due modi di lavorare, ed è bene capire la differenza fin dall'inizio:

- **Scripting layer** (`plt.plot()`, o direttamente `df.plot()` da pandas): è come scrivere un appunto veloce a mano. Dici "disegna questo" e matplotlib decide per te il resto — quale figura usare, dove metterla. È veloce e perfetto per esplorare i dati.
- **Artist layer** (`fig, ax = plt.subplots()`): è come progettare con livelli sovrapposti in un programma di design, dove controlli esplicitamente ogni elemento — la tela (`fig`), gli assi (`ax`), cosa viene disegnato in ciascuno. Richiede più codice, ma ti dà il controllo totale, specialmente quando vuoi più grafici nella stessa figura.

Entrambi gli approcci producono risultati validi. Vedremo esempi di entrambi durante la lezione.

## Preparare l'ambiente e caricare i dati

Carichiamo il dataset dell'immigrazione verso il Canada (lo stesso usato nella lezione su pandas), lo indicizziamo per paese e prepariamo la lista degli anni.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

%matplotlib inline

print('Matplotlib version: ', mpl.__version__)

In [ ]:
URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Canada.csv'

df_can = pd.read_csv(URL)

print('Data read into a pandas dataframe!')
df_can.head()

Come abbiamo visto nella lezione precedente, conviene avere il DataFrame indicizzato per paese e la lista degli anni come testo pronta per essere usata in qualsiasi grafico.

In [ ]:
df_can.set_index('Country', inplace=True)
df_can.index.name = None

years = list(map(str, range(1980, 2014)))

df_can.head(3)

## Line plot di un solo paese (una Serie)

Iniziamo con un caso reale: nel 2010, Haiti subì un devastante terremoto di magnitudo 7.0. Come parte della sua risposta umanitaria, il Canada accettò molti più rifugiati haitiani quell'anno. Visualizziamolo.

Per prima cosa estraiamo i dati di Haiti come una **Serie** (una sola riga, con gli anni come indice):

In [ ]:
haiti = df_can.loc['Haiti', years]  # serie: indice = anni, valori = numero di immigrati
haiti.head()

Con l'approccio dello scripting layer, disegnare una Serie è semplice come chiamare `.plot()`. `pandas` popola automaticamente l'asse X con i valori dell'indice (gli anni) e l'asse Y con i valori della colonna.

In [ ]:
haiti.plot(kind='line')

plt.title('Immigrazione da Haiti')
plt.ylabel('Numero di immigrati')
plt.xlabel('Anni')

plt.show()  # necessario per mostrare le modifiche fatte alla figura

Gli anni sono salvati come testo (`str`), il che può influenzare come appaiono i tick dell'asse X. Per un risultato più pulito, conviene convertire l'indice in intero prima di creare il grafico. Ne approfittiamo per annotare il terremoto del 2010 direttamente sul grafico con `plt.text(x, y, etichetta)`.

In [ ]:
haiti.index = haiti.index.map(int)
haiti.plot(kind='line')

plt.title('Immigrazione da Haiti')
plt.ylabel('Numero di immigrati')
plt.xlabel('Anni')

# annotiamo il terremoto del 2010 direttamente sul grafico
# sintassi: plt.text(x, y, etichetta)
plt.text(2000, 6000, '2010 Earthquake')

plt.show()

**Nota su `plt.text(x, y, etichetta)`**: poiché l'asse X (anni) ora è di tipo intero, `x` si specifica come un anno reale (ad esempio, 2000). Se l'indice fosse ancora testo, dovremmo usare la posizione dell'anno nella lista invece dell'anno stesso.

## Line plot con più paesi (un DataFrame)

Ora confrontiamo l'immigrazione da India e Cina tra il 1980 e il 2013.

In [ ]:
df_CI = df_can.loc[['India', 'China'], years]
df_CI

Se proviamo a creare il grafico di `df_CI` direttamente con `.plot(kind='line')`, il risultato non è buono. Il motivo: pandas disegna i **valori dell'indice sull'asse X** e **ogni colonna come una linea distinta**. Poiché `df_CI` ha i paesi nell'indice e gli anni nelle colonne, pandas cerca di disegnare una linea per ogni anno — esattamente il contrario di ciò che vogliamo.

La soluzione è **trasporre** il DataFrame con `.transpose()`, che scambia righe con colonne.

In [ ]:
df_CI = df_CI.transpose()
df_CI.head()

Ora gli anni sono nell'indice (asse X) e ogni paese è una colonna (una linea). Convertiamo l'indice in intero e creiamo il grafico.

In [ ]:
df_CI.index = df_CI.index.map(int)
df_CI.plot(kind='line')

plt.title('Immigrati da Cina e India')
plt.ylabel('Numero di immigrati')
plt.xlabel('Anni')

plt.show()

**Perché non abbiamo dovuto trasporre `haiti` prima?** Perché `haiti` è una **Serie**, non un DataFrame: ha già gli anni come indice in modo naturale, con un unico valore per anno. Un DataFrame invece ha righe e colonne, e `.plot()` assume sempre che l'indice vada sull'asse X e che ogni colonna diventi una linea — per questo devi trasporre solo quando le tue "serie da confrontare" sono organizzate come righe invece che come colonne.

Come buona pratica: evita di mettere più di 5-10 linee nello stesso grafico a linee. Oltre quel numero, diventa difficile da leggere.

## Top 5 paesi con più immigrazione

Combiniamo `sort_values()`, `.head()` e `.transpose()` per confrontare la tendenza dei 5 paesi che hanno contribuito di più all'immigrazione canadese.

In [ ]:
df_top5 = df_can.sort_values(by='Total', ascending=False, axis=0).head(5)
df_top5 = df_top5[years].transpose()
df_top5.index = df_top5.index.map(int)

df_top5.plot(kind='line', figsize=(14, 8))

plt.title('Tendenza di immigrazione dei 5 paesi principali')
plt.ylabel('Numero di immigrati')
plt.xlabel('Anni')

plt.show()

## L'equivalente nell'artist layer

Lo stesso grafico a linee, ma controllando esplicitamente figura e assi con `fig, ax = plt.subplots()`.

In [ ]:
# creiamo la figura e gli assi
fig, ax = plt.subplots()

total_immigrants = df_can[years].sum()          # totale di immigrati per anno
total_immigrants.index = total_immigrants.index.map(int)

ax.plot(total_immigrants)

ax.set_title('Immigrati tra il 1980 e il 2013')
ax.set_xlabel('Anni')
ax.set_ylabel('Totale immigrati')

plt.show()

Con `ax.plot()` possiamo personalizzare la linea con più dettaglio: marcatore, dimensione, colore e stile della linea.

In [ ]:
fig, ax = plt.subplots()

ax.plot(total_immigrants,
        marker='s',           # marcatori a forma di quadrato
        markersize=5,         # dimensione del marcatore
        color='green',        # colore della linea
        linestyle='dotted')   # linea punteggiata

ax.set_title('Immigrati tra il 1980 e il 2013')
ax.set_xlabel('Anni')
ax.set_ylabel('Totale immigrati')
ax.legend(['Immigrati'])

plt.show()

## Area plot

Il grafico ad area è praticamente un grafico a linee con lo spazio sotto la curva riempito di colore, il che aiuta a comunicare il volume accumulato invece della sola tendenza. In pandas si usa `kind='area'`.

Di default, `pandas` impila le aree una sull'altra (`stacked=True`), così il totale visivo in ogni punto è la somma di tutte le serie.

In [ ]:
df_top5.plot(kind='area', alpha=0.5, figsize=(14, 8))  # alpha controlla la trasparenza, stacked=True di default

plt.title('Tendenza di immigrazione dei 5 paesi principali (impilato)')
plt.ylabel('Numero di immigrati')
plt.xlabel('Anni')

plt.show()

Se preferiamo vedere ogni area sovrapposta da zero (per confrontare le tendenze individuali senza che si sommino), usiamo `stacked=False`. In quel caso, `alpha` è particolarmente utile perché le aree si sovrappongono e dobbiamo poter vedere attraverso di esse.

In [ ]:
df_top5.plot(kind='area', stacked=False, alpha=0.5, figsize=(14, 8))

plt.title('Tendenza di immigrazione dei 5 paesi principali (senza impilare)')
plt.ylabel('Numero di immigrati')
plt.xlabel('Anni')

plt.show()

Nell'artist layer, l'equivalente diretto di un'area è `ax.fill_between(x, y1, y2, color=..., alpha=...)`, che riempie lo spazio tra due curve (o tra una curva e l'asse X se `y2=0`).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.fill_between(total_immigrants.index, total_immigrants, color='skyblue', alpha=0.4)
ax.plot(total_immigrants, color='steelblue')

ax.set_title('Totale immigrati tra il 1980 e il 2013 (area con fill_between)')
ax.set_xlabel('Anni')
ax.set_ylabel('Totale immigrati')
ax.grid(True)

plt.show()

## Personalizzazione base che userai sempre

Hai già visto diversi di questi elementi negli esempi precedenti. Eccoli riuniti come riferimento rapido:

```python
plt.title('Titolo del grafico')      # o ax.set_title(...)
plt.xlabel('Etichetta asse X')       # o ax.set_xlabel(...)
plt.ylabel('Etichetta asse Y')       # o ax.set_ylabel(...)
plt.legend(['Serie 1', 'Serie 2'])   # o ax.legend([...])
plt.grid(True)                       # o ax.grid(True) — aggiunge una griglia di sfondo
plt.text(x, y, 'Annotazione')        # o ax.annotate('Annotazione', xy=(x, y)) — testo su un punto specifico
```

Con l'artist layer, `ax.set_xlim()` ti permette di regolare i limiti dell'asse X, e `ax.set_xticks(...)` controlla esattamente quali marcature appaiono su di esso — utile quando l'asse di default appare saturo o poco chiaro.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(total_immigrants, marker='o', markersize=4, color='darkorange')

ax.set_title('Immigrati tra il 1980 e il 2013')
ax.set_xlabel('Anni')
ax.set_ylabel('Totale immigrati')
ax.legend(['Immigrati'])
ax.grid(True)
ax.set_xlim(1978, 2015)
ax.set_xticks(list(range(1980, 2015, 5)))

plt.show()

Questo stesso dataset e questa stessa libreria servono anche per grafici a dispersione, barre, istogrammi e pie chart — ma questi tipi di grafico sono trattati in dettaglio in un'altra lezione dedicata a confrontare categorie e distribuzioni, non qui.

## Punti chiave

- Lo scripting layer (`plt.plot()`, `df.plot()`) è veloce per esplorare; l'artist layer (`fig, ax = plt.subplots()`) ti dà il controllo totale, specialmente con più grafici.
- Una Serie ha già l'asse X naturale nel suo indice; un DataFrame ha bisogno che le categorie da confrontare siano nelle colonne, per questo a volte bisogna `.transpose()` prima di creare il grafico.
- Convertire l'indice degli anni in intero (`.map(int)`) evita problemi di visualizzazione sull'asse X.
- L'area plot è un line plot con riempimento sotto la curva; `stacked` decide se le aree si sommano o si sovrappongono, e `alpha` controlla la trasparenza.
- `plt.title`, `plt.xlabel`, `plt.ylabel`, `plt.legend`, `plt.grid` e `plt.text`/`ax.annotate` sono gli elementi di personalizzazione che userai in quasi ogni grafico.